# Part 8 — One number to train on: the ELBO for diffusion

_Rigorous Courses · Diffusion Models — Part 8 of 12_

**The whole reverse chain gets scored by one trainable number, and it splits into per-step Gaussian comparisons**

You will build a 3-state diffusion model small enough to enumerate every path, compute its exact log-likelihood AND its ELBO side by side, watch the gap between them close when the reverse model is set to the truth, and see that in the Gaussian case the whole training signal reduces to one job: hit the true posterior mean. Every hand calculation from the lesson gets verified here.

---

This notebook accompanies the lesson. Run cells top to bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)

## The generative model, on a toy you can enumerate

The lesson's whole derivation works for any Markov chain — nothing needs the bell curve until the very last section. So we build the smallest possible diffusion model: a one-pixel world with **3 states** (brightness levels 0, 1, 2) and $T = 2$ noising steps. A hidden path is the pair $(x_1, x_2)$, so there are only $3 \times 3 = 9$ paths — few enough to enumerate every single one and compute every probability exactly. This is the lesson's toy trimmed from $T = 3$ to $T = 2$ so all the paths fit on one screen; the lesson's hand-worked numbers carry over exactly, because they only ever involved the two steps $x_0 \to x_1 \to x_2$.

The forward "noising" kernel is the same at every step: a state **keeps** its value with probability 0.7 and hops to each of the other two states with probability 0.15.

### Step 1 — Build the forward kernel and watch it forget

The kernel is a 3-by-3 table `K` where `K[i, j]` is the probability of hopping from state $i$ to state $j$. Each row is a full probability table (part 2), so rows must sum to 1. We also pick a data distribution `p_data` over the clean state — deliberately lopsided, so nothing later is trivial — and push it through the kernel to get the marginals $q(x_1)$ and $q(x_2)$.

Applying the kernel over and over shows the discrete cousin of part 6's end state: whatever the start, the marginal drifts toward uniform $(1/3, 1/3, 1/3)$. That is exactly why the reverse chain may **fix** its starting distribution instead of learning it. But watch the printout closely: after only our $T = 2$ gentle steps the chain has *not* fully forgotten — $q(x_2)$ is still visibly lopsided. That honest imperfection will show up later as a non-tiny $L_T$, and Step 13 contrasts it with the real schedule's microscopic one.

In [ ]:
K = np.array([
    [0.70, 0.15, 0.15],
    [0.15, 0.70, 0.15],
    [0.15, 0.15, 0.70],
])
p_data = np.array([0.6, 0.3, 0.1])

q1 = p_data @ K
q2 = q1 @ K

marg10 = p_data.copy()
for step in range(10):
    marg10 = marg10 @ K

print(f"q(x1)                   = {np.round(q1, 4)}")
print(f"q(x2)                   = {np.round(q2, 4)}")
print(f"marginal after 10 steps = {np.round(marg10, 4)}   (uniform = 0.3333)")

assert np.allclose(K.sum(axis=1), 1.0), "each row of the kernel is a probability table"
assert np.allclose(q1.sum(), 1.0)
assert np.allclose(q2.sum(), 1.0)
assert np.max(np.abs(marg10 - 1.0 / 3.0)) < 0.01, "10 steps in, the chain has all but forgotten its start"

### Step 2 — Define a candidate reverse model

The model generates in reverse: draw $x_2$ from a fixed start, apply a learned step down to $x_1$, then decode to $x_0$. That is the lesson's generative chain with $T = 2$:

$$p_\theta(x_0, x_1, x_2) = p(x_2)\; p_\theta(x_1 \mid x_2)\; p_\theta(x_0 \mid x_1)$$

Our candidate model, before any training: fix the start at uniform (the discrete stand-in for pure noise), let the middle step shrug uniformly — exactly the guess the lesson's hand example scores — and give the decode step a mild preference for staying put. `R1[j]` is the row $p_\theta(x_1 \mid x_2 = j)$ and `R0[i]` is the row $p_\theta(x_0 \mid x_1 = i)$.

In [ ]:
p_start = np.full(3, 1.0 / 3.0)
R1 = np.full((3, 3), 1.0 / 3.0)
R0 = np.array([
    [0.50, 0.25, 0.25],
    [0.25, 0.50, 0.25],
    [0.25, 0.25, 0.50],
])

print(f"p(x2) fixed start          = {np.round(p_start, 4)}")
print("p_theta(x1 | x2) rows      = uniform (1/3, 1/3, 1/3) for every x2")
print("p_theta(x0 | x1) rows:")
print(R0)

assert np.allclose(R1.sum(axis=1), 1.0)
assert np.allclose(R0.sum(axis=1), 1.0)

### Step 3 — Exact log-likelihood by enumerating all 9 paths

The number training cares about is the marginal $p_\theta(x_0)$: the total probability, accumulated over every possible hidden path, of the whole journey. For images that is a hopeless integral over hundreds of thousands of linked numbers. Here it is a sum of 9 products:

$$p_\theta(x_0) = \sum_{x_1} \sum_{x_2} p(x_2)\; p_\theta(x_1 \mid x_2)\; p_\theta(x_0 \mid x_1)$$

We score the clean point $x_0 = 0$. This exact value is our gold standard — the thing the ELBO is supposed to approach from below.

In [ ]:
x0 = 0

p_x0 = 0.0
for x1 in range(3):
    for x2 in range(3):
        p_path = p_start[x2] * R1[x2, x1] * R0[x1, x0]
        p_x0 = p_x0 + p_path
log_px0 = np.log(p_x0)

print(f"p_theta(x0 = 0)     = {p_x0:.6f}")
print(f"log p_theta(x0 = 0) = {log_px0:.6f} nats")

## Borrow the ELBO: the forward chain is our q

Part 5's template needs a helper distribution over the hidden variable. Diffusion's helper is the fixed forward chain itself — here, two applications of the kernel `K`, with zero learnable parameters. The bound is an average of a log-ratio of path scores:

$$\mathrm{ELBO}(x_0) = \mathbb{E}_{q(x_{1:T} \mid x_0)}\big[\log p_\theta(x_{0:T}) - \log q(x_{1:T} \mid x_0)\big]$$

In words: run the forward noising chain on $x_0$, score the resulting path under the model and under the chain itself, and average the difference. For images that average needs sampling; for our toy the expectation is an exact sum over the same 9 paths.

### Step 4 — Compute the ELBO exactly, and check the bound direction

Each path contributes its forward-chain probability $q(x_1, x_2 \mid x_0) = K[x_0, x_1]\, K[x_1, x_2]$ times the log-ratio of the model's path score to the chain's path score. The assert is part 5's theorem: the ELBO may never exceed the true log-likelihood — for any model, any data point.

In [ ]:
elbo = 0.0
for x1 in range(3):
    for x2 in range(3):
        q_path = K[x0, x1] * K[x1, x2]
        log_p_joint = np.log(p_start[x2] * R1[x2, x1] * R0[x1, x0])
        log_q_path = np.log(q_path)
        elbo = elbo + q_path * (log_p_joint - log_q_path)

gap = log_px0 - elbo

print(f"ELBO(x0 = 0)        = {elbo:.6f} nats")
print(f"log p_theta(x0 = 0) = {log_px0:.6f} nats")
print(f"gap                 = {gap:.6f} nats")

assert elbo <= log_px0 + 1e-12, "the ELBO may never exceed the log-likelihood"

### Step 5 — The gap is exactly a KL

Part 5's identity says the gap is not merely some positive leftover — it is precisely

$$\log p_\theta(x_0) - \mathrm{ELBO}(x_0) = D_{\mathrm{KL}}\big(q(x_{1:T} \mid x_0)\,\big\|\,p_\theta(x_{1:T} \mid x_0)\big),$$

the mismatch between the forward chain's distribution over hidden paths and the model's own posterior over hidden paths. Both are 9-entry tables here — the model's posterior is each path's joint score divided by $p_\theta(x_0)$ (a conditional is the renormalized slice, part 2) — so we can compute the KL outright and demand it equals the gap to machine precision.

In [ ]:
q_paths = np.zeros((3, 3))
p_paths = np.zeros((3, 3))
for x1 in range(3):
    for x2 in range(3):
        q_paths[x1, x2] = K[x0, x1] * K[x1, x2]
        p_paths[x1, x2] = p_start[x2] * R1[x2, x1] * R0[x1, x0] / p_x0

kl_paths = np.sum(q_paths * np.log(q_paths / p_paths))

print(f"KL(forward chain || model posterior) = {kl_paths:.6f} nats")
print(f"gap from Step 4                      = {gap:.6f} nats")

assert np.allclose(q_paths.sum(), 1.0)
assert np.allclose(p_paths.sum(), 1.0)
assert abs(kl_paths - gap) < 1e-12, "the gap IS the KL: part 5's identity, to machine precision"

## The Bayes flip and the telescope

The lesson's decomposition leans on one crucial cancellation. After the Bayes flip, the awkward forward-step scores leave behind a sum of marginal ratios, and that sum **telescopes**:

$$\sum_{t=2}^{T} \log\frac{q(x_t \mid x_0)}{q(x_{t-1} \mid x_0)} = \log\frac{q(x_T \mid x_0)}{q(x_1 \mid x_0)}$$

Every intermediate marginal is added by one term and subtracted by the next, so only the last top and the first bottom survive. The lesson showed this with symbols; let's watch it happen with real numbers.

### Step 6 — Print every ratio and watch the middle terms vanish

We run the toy chain out to $T = 3$ (one step more than the ELBO toy, so there is a genuine middle to cancel), sample three noisy paths from $x_0 = 0$ with the seeded generator, and evaluate the log-marginals $q_t = \log q(x_t \mid x_0)$ at each path's actual states. For every path, the two ratio terms $(q_2 - q_1)$ and $(q_3 - q_2)$ must sum to the collapsed single ratio $q_3 - q_1$: the $+q_2$ from one bracket meets the $-q_2$ from the next and vanishes. For the real thing, $T = 1000$, the same move collapses 999 awkward ratios into one.

In [ ]:
marg1 = K[x0]
marg2 = K[x0] @ K
marg3 = marg2 @ K

for trial in range(3):
    s1 = rng.choice(3, p=K[x0])
    s2 = rng.choice(3, p=K[s1])
    s3 = rng.choice(3, p=K[s2])
    lq1 = np.log(marg1[s1])
    lq2 = np.log(marg2[s2])
    lq3 = np.log(marg3[s3])
    term_t2 = lq2 - lq1
    term_t3 = lq3 - lq2
    total = term_t2 + term_t3
    collapsed = lq3 - lq1

    print(f"path x1,x2,x3 = {s1},{s2},{s3}:   (q2 - q1) = {term_t2:+.4f}   (q3 - q2) = {term_t3:+.4f}")
    print(f"   sum of ratios = {total:+.4f}   collapsed q3 - q1 = {collapsed:+.4f}")

    assert abs(total - collapsed) < 1e-12, "the telescope collapses exactly"

## Three kinds of terms: naming the pieces

After the flip, the telescope, and the $t = 1$ cancellation, the lesson lands on the most important equation of this part:

$$-\mathrm{ELBO}(x_0) \;=\; L_T \;+\; \sum_{t=2}^{T} L_{t-1} \;+\; L_0$$

$$L_T = D_{\mathrm{KL}}\big(q(x_T \mid x_0) \,\|\, p(x_T)\big) \qquad L_{t-1} = \mathbb{E}_{q(x_t \mid x_0)}\, D_{\mathrm{KL}}\big(q(x_{t-1} \mid x_t, x_0) \,\|\, p_\theta(x_{t-1} \mid x_t)\big) \qquad L_0 = -\,\mathbb{E}_{q(x_1 \mid x_0)} \log p_\theta(x_0 \mid x_1)$$

For our $T = 2$ toy the middle sum has a single term, $L_1$: the true one-step denoiser $q(x_1 \mid x_2, x_0)$ against the learned $p_\theta(x_1 \mid x_2)$, averaged over where $x_2$ lands. Before assembling it, we verify the lesson's hand-worked inner KL.

### Step 7 — Verify the lesson's hand-worked posterior and KL

The lesson computed, for a chain started at $x_0 = 0$ and observed at $x_2 = 2$, the true posterior over the midpoint and its KL against the uniform guess:

$$q(x_1 \mid x_2 = 2,\ x_0 = 0) = (0.4516,\ 0.0968,\ 0.4516) \qquad D_{\mathrm{KL}}\big(q \,\|\, \mathrm{uniform}\big) \approx 0.1546 \ \mathrm{nats}$$

Same four moves in code: forward reach (row 0 of the kernel), backward look (column 2), multiply elementwise (Bayes' numerator: reach times look), normalize (a conditional is the renormalized slice). The asserts demand the lesson's 4-decimal values.

In [ ]:
reach = K[0]
look = K[:, 2]
numer = reach * look
posterior = numer / numer.sum()

kl_hand = np.sum(posterior * np.log(posterior / R1[2]))

print(f"forward reach  q(x1 | x0=0)      = {np.round(reach, 4)}")
print(f"backward look  q(x2=2 | x1)      = {np.round(look, 4)}")
print(f"posterior q(x1 | x2=2, x0=0)     = {np.round(posterior, 4)}")
print(f"KL(posterior || uniform guess)   = {kl_hand:.4f} nats   (lesson: 0.1546)")

assert np.allclose(posterior, [0.4516, 0.0968, 0.4516], atol=1e-4)
assert abs(kl_hand - 0.1546) < 5e-4

### Step 8 — Assemble L_T, L_1, L_0 and match the ELBO

Now each named piece by its definition: $L_T$ compares where the forward chain actually ends against the fixed uniform start; $L_1$ averages Step 7's kind of inner KL over all three places $x_2$ can land, weighted by how often each occurs; $L_0$ is the average surprise of the decode step at the true clean point. The acid test: their sum must equal $-\mathrm{ELBO}$ from Step 4 **exactly** — the decomposition is an identity, not an approximation.

Read the bar chart afterwards: for this untrained candidate the decode term dominates, and $L_T$ is a real cost (about 0.086 nats) because our two gentle steps never finished destroying the signal. And remember the lesson's point: $L_T$ contains no $\theta$ — training cannot change it, so it gets dropped from the loss.

In [ ]:
q2_given0 = K[x0] @ K
L_T = np.sum(q2_given0 * np.log(q2_given0 / p_start))

L_1 = 0.0
for x2 in range(3):
    numer_x2 = K[x0] * K[:, x2]
    post_x2 = numer_x2 / numer_x2.sum()
    kl_x2 = np.sum(post_x2 * np.log(post_x2 / R1[x2]))
    L_1 = L_1 + q2_given0[x2] * kl_x2

q1_given0 = K[x0]
L_0 = -np.sum(q1_given0 * np.log(R0[:, x0]))

total_loss = L_T + L_1 + L_0

print(f"L_T             = {L_T:.6f} nats   (start mismatch — no theta inside)")
print(f"L_1             = {L_1:.6f} nats   (the middle training signal)")
print(f"L_0             = {L_0:.6f} nats   (the decode term)")
print(f"L_T + L_1 + L_0 = {total_loss:.6f}")
print(f"-ELBO           = {-elbo:.6f}")

assert abs(total_loss - (-elbo)) < 1e-12, "the decomposition is exact, term by term"

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.bar(["L_T", "L_1", "L_0"], [L_T, L_1, L_0], color=["#999999", "#4ea1ff", "#ff7b72"])
ax.set_xlabel("loss piece")
ax.set_ylabel("nats")
ax.set_title("-ELBO split into its three named pieces (candidate model)")
plt.tight_layout()
plt.show()

## Tight when the model is right

The lesson's sanity checks made two promises. One: the bound can never exceed the log-likelihood (Step 4 verified it). Two: the bound is **tight** — gap exactly zero — when the reverse model is right. Let's collect on the second promise, then break the model on purpose and watch the gap grow.

### Step 9 — Set the reverse model to the truth and watch the gap close

"Right" means: every reverse step equals the true reverse conditional of the forward chain, and the start equals the chain's true end marginal. The true reverse conditionals come from Bayes' rule on the marginals (part 2, and part 4's weather-chain flip):

$$q(x_{t-1} = i \mid x_t = j) = \frac{q_{t-1}(i)\; K[i, j]}{q_t(j)}$$

Note these condition on nothing but $x_t$ — they average over the data distribution, no cheating with $x_0$. If the lesson's Check 2 is honest, this model's log-likelihood must equal $\log p_{\mathrm{data}}(x_0)$ on the nose, and the gap must close to zero. The asserts demand ten decimal places.

In [ ]:
R1_true = np.zeros((3, 3))
R0_true = np.zeros((3, 3))
for j in range(3):
    R1_true[j] = q1 * K[:, j] / q2[j]
    R0_true[j] = p_data * K[:, j] / q1[j]
p_start_true = q2


def elbo_and_loglik(p_start_m, R1_m, R0_m, x0_state):
    p_marg = 0.0
    bound = 0.0
    for x1 in range(3):
        for x2 in range(3):
            p_path = p_start_m[x2] * R1_m[x2, x1] * R0_m[x1, x0_state]
            p_marg = p_marg + p_path
    for x1 in range(3):
        for x2 in range(3):
            q_path = K[x0_state, x1] * K[x1, x2]
            log_p_joint = np.log(p_start_m[x2] * R1_m[x2, x1] * R0_m[x1, x0_state])
            bound = bound + q_path * (log_p_joint - np.log(q_path))
    return bound, np.log(p_marg)


elbo_true, loglik_true = elbo_and_loglik(p_start_true, R1_true, R0_true, x0)
gap_true = loglik_true - elbo_true

print(f"log p_theta(x0=0), true reverse = {loglik_true:.10f}")
print(f"log p_data(x0=0)                = {np.log(p_data[0]):.10f}")
print(f"gap                             = {gap_true:.2e}")

assert np.allclose(R1_true.sum(axis=1), 1.0)
assert np.allclose(R0_true.sum(axis=1), 1.0)
assert abs(loglik_true - np.log(p_data[0])) < 1e-10, "a right model recovers the data probability exactly"
assert abs(gap_true) < 1e-10, "the bound is tight when the model is right"

### Step 10 — Dial the wrongness and watch the gap grow

Now interpolate every table of the model between the truth ($w = 0$) and the shrugging candidate ($w = 1$). Each blend is still a valid model — a mixture of probability rows is a probability row. The gap should start at zero, stay nonnegative (it is a KL), and grow as the model gets more wrong. This curve is the ELBO's promise in one picture: pushing the bound up is the same thing as dragging the model toward the truth.

In [ ]:
w_grid = np.linspace(0.0, 1.0, 11)
gaps = []
for w in w_grid:
    p_start_w = (1.0 - w) * p_start_true + w * p_start
    R1_w = (1.0 - w) * R1_true + w * R1
    R0_w = (1.0 - w) * R0_true + w * R0
    elbo_w, loglik_w = elbo_and_loglik(p_start_w, R1_w, R0_w, x0)
    gaps.append(loglik_w - elbo_w)
gaps = np.array(gaps)

print("w   :", np.round(w_grid, 2))
print("gap :", np.round(gaps, 4))

plt.figure(figsize=(6.5, 4))
plt.plot(w_grid, gaps, "-o", color="#4ea1ff")
plt.xlabel("w — how far the reverse model sits from the truth")
plt.ylabel("gap = log p(x0) - ELBO  (nats)")
plt.title("the gap is zero at the truth and grows with wrongness")
plt.show()

assert gaps[0] < 1e-10
assert np.all(gaps > -1e-12), "a gap is a KL: it can never go negative"
assert gaps[-1] > gaps[0] + 0.05, "a wrong model pays a visible gap"

## When both sides are Gaussian: hit the mean

Back to the real, continuous diffusion. Inside each $L_{t-1}$ sit two Gaussians — part 7's true denoiser $\mathcal{N}(\tilde\mu_t, \tilde\beta_t)$ and the learned $\mathcal{N}(\mu_\theta(x_t, t), \sigma_t^2)$ — so part 5's closed-form Gaussian KL applies and the lesson reduced the whole per-step cost to:

$$L_{t-1} = \frac{1}{2\sigma_t^2}\, \mathbb{E}_{q}\,\big\|\tilde\mu_t(x_t, x_0) - \mu_\theta(x_t, t)\big\|^2 + \mathrm{const}$$

Hitting the true posterior mean is the whole game. Let's see that literally, with the course's running numbers.

### Step 11 — The running example's true posterior at t = 2

Recall the shared hand example: $T = 3$, $\beta = (0.1,\ 0.2,\ 0.3)$, so $\alpha = (0.9,\ 0.8,\ 0.7)$ and $\bar\alpha = (0.9,\ 0.72,\ 0.504)$. Part 7 computed, for $x_0 = 2$ observed at $x_2 = 1.1$:

$$\tilde\mu_2 = 1.7066 \qquad \tilde\beta_2 = 0.0714$$

We recompute both from part 7's formulas — $\tilde\beta_t = (1-\bar\alpha_{t-1})\beta_t/(1-\bar\alpha_t)$ and $\tilde\mu_t$ as the weighted average of $x_0$ and $x_t$ — and assert the 4-decimal values.

In [ ]:
betas = np.array([0.1, 0.2, 0.3])
alphas = 1.0 - betas
abar = np.cumprod(alphas)

x0_val = 2.0
x2_val = 1.1

beta_tilde = (1.0 - abar[0]) * betas[1] / (1.0 - abar[1])
weight_x0 = np.sqrt(abar[0]) * betas[1] / (1.0 - abar[1])
weight_xt = np.sqrt(alphas[1]) * (1.0 - abar[0]) / (1.0 - abar[1])
mu_tilde = weight_x0 * x0_val + weight_xt * x2_val

print(f"weight on x0 = {weight_x0:.6f}   weight on x2 = {weight_xt:.6f}")
print(f"mu_tilde_2   = {mu_tilde:.4f}   (lesson: 1.7066)")
print(f"beta_tilde_2 = {beta_tilde:.4f}   (lesson: 0.0714)")

assert abs(mu_tilde - 1.7066) < 1e-4
assert abs(beta_tilde - 0.0714) < 5e-5

### Step 12 — The KL parabola: the minimum sits at the posterior mean

Fix the learned variance to the matched choice $\sigma_t^2 = \tilde\beta_t$ and sweep the network's guessed mean $\mu_\theta$ across a grid. Part 5's Gaussian KL formula prices every guess, and with matched variances the lesson showed the price collapses to the clean special case

$$D_{\mathrm{KL}} = \frac{(\tilde\mu_t - \mu_\theta)^2}{2\tilde\beta_t}$$

— a parabola in the guess, worth exactly zero at $\mu_\theta = \tilde\mu_t$. We verify three ways: the general formula matches the shortcut everywhere on the grid, the parabola's minimum sits at $\tilde\mu_2$, and one value is confirmed by brute-force numerical integration — "no integrals, ever" is a theorem, not a hope.

In [ ]:
def gauss_kl(m1, v1, m2, v2):
    return np.log(np.sqrt(v2 / v1)) + (v1 + (m1 - m2) ** 2) / (2.0 * v2) - 0.5


mu_grid = np.linspace(0.5, 3.0, 251)
kl_curve = gauss_kl(mu_tilde, beta_tilde, mu_grid, beta_tilde)
shortcut = (mu_tilde - mu_grid) ** 2 / (2.0 * beta_tilde)

guess = 1.5
kl_closed = gauss_kl(mu_tilde, beta_tilde, guess, beta_tilde)
half_width = 8.0 * np.sqrt(beta_tilde)
xs = np.linspace(mu_tilde - half_width, mu_tilde + half_width, 20001)
dx = xs[1] - xs[0]
q_pdf = np.exp(-(xs - mu_tilde) ** 2 / (2.0 * beta_tilde)) / np.sqrt(2.0 * np.pi * beta_tilde)
p_pdf = np.exp(-(xs - guess) ** 2 / (2.0 * beta_tilde)) / np.sqrt(2.0 * np.pi * beta_tilde)
kl_numeric = np.sum(q_pdf * np.log(q_pdf / p_pdf)) * dx

best_mu = mu_grid[np.argmin(kl_curve)]

print(f"KL at guess 1.5: closed formula = {kl_closed:.6f}   numeric integral = {kl_numeric:.6f}")
print(f"parabola minimum at mu = {best_mu:.3f}   (mu_tilde = {mu_tilde:.4f})")

plt.figure(figsize=(6.5, 4))
plt.plot(mu_grid, kl_curve, color="#4ea1ff", label="KL(true posterior || guess)")
plt.axvline(mu_tilde, color="#ff7b72", linestyle="--", label="mu_tilde")
plt.xlabel("guessed mean mu_theta")
plt.ylabel("KL (nats)")
plt.title("matched variances: the per-step cost is a parabola in the guessed mean")
plt.legend()
plt.show()

assert np.allclose(kl_curve, shortcut, atol=1e-12), "matched variances: KL = squared mean gap / (2 var)"
assert abs(best_mu - mu_tilde) < 0.006, "the minimum sits at the true posterior mean"
assert abs(kl_numeric - kl_closed) < 1e-6, "the closed formula agrees with the brute-force integral"

## Sanity checks in numbers

One promise is still unpriced: the lesson claimed that for the **real** schedule, dropping $L_T$ costs about $8 \times 10^{-5}$ nats — so small that fixing the start at pure noise is essentially free. Our toy's $L_T$ was a thousand times bigger. Let's compute the real number.

### Step 13 — Price L_T on the real schedule

Build the $T = 1000$ linear schedule from part 6, form the true end distribution for a coordinate with $x_0 = 2$ — the closed form gives $q(x_T \mid x_0) = \mathcal{N}(\sqrt{\bar\alpha_T}\, x_0,\ 1-\bar\alpha_T)$ — and plug it into part 5's Gaussian KL against the fixed start $\mathcal{N}(0, 1)$. The asserts pin the lesson's claim.

In [ ]:
T_real = 1000
betas_real = np.linspace(1e-4, 0.02, T_real)
abar_T = np.prod(1.0 - betas_real)

mean_T = np.sqrt(abar_T) * 2.0
var_T = 1.0 - abar_T
L_T_real = gauss_kl(mean_T, var_T, 0.0, 1.0)

print(f"abar_T on the real schedule = {abar_T:.2e}")
print(f"q(x_T | x0 = 2) = N({mean_T:.4f}, {var_T:.5f})")
print(f"L_T on the real schedule    = {L_T_real:.2e} nats   (lesson: about 8e-5)")
print(f"our toy's L_T was {L_T:.4f} nats — about {L_T / L_T_real:,.0f} times larger, the price of a short, gentle chain")

assert L_T_real < 1e-4, "dropping L_T on the real schedule costs less than a ten-thousandth of a nat"
assert abs(L_T_real - 8e-5) < 2e-5

## Practice

Try each one in the empty cell below it, then reveal the worked solution. These are the same problems as the lesson — redo them here with code as your calculator and checker.

**Problem 1.** Write out the telescope sum $\sum_{t=2}^{4} \log\dfrac{q(x_t \mid x_0)}{q(x_{t-1} \mid x_0)}$ for $T = 4$, term by term, and show it collapses to $\log\dfrac{q(x_4 \mid x_0)}{q(x_1 \mid x_0)}$. Use the shorthand $q_t$ for $\log q(x_t \mid x_0)$. Then confirm with numbers: run the toy chain to $T = 4$ and evaluate the sum along the path $x_1, x_2, x_3, x_4 = 0, 1, 1, 2$ from $x_0 = 0$.

In [ ]:
# Your turn:

<details><summary>Show worked solution</summary>

- Substitute $t = 2, 3, 4$: the three terms are $\log\frac{q(x_2 \mid x_0)}{q(x_1 \mid x_0)} + \log\frac{q(x_3 \mid x_0)}{q(x_2 \mid x_0)} + \log\frac{q(x_4 \mid x_0)}{q(x_3 \mid x_0)}$.
- Split each log of a quotient into a difference: $(q_2 - q_1) + (q_3 - q_2) + (q_4 - q_3)$.
- Cancel the $+q_2$ against the $-q_2$, then the $+q_3$ against the $-q_3$: the same number with opposite signs sums to zero.
- Collect the survivors: $q_4 - q_1$, and recombine: $\log\frac{q(x_4 \mid x_0)}{q(x_1 \mid x_0)}$.

```python
m1 = K[0]
m2 = m1 @ K
m3 = m2 @ K
m4 = m3 @ K
lq1 = np.log(m1[0])
lq2 = np.log(m2[1])
lq3 = np.log(m3[1])
lq4 = np.log(m4[2])
terms = np.array([lq2 - lq1, lq3 - lq2, lq4 - lq3])

print("terms:", np.round(terms, 4))
print(f"sum = {terms.sum():.4f}   collapsed q4 - q1 = {lq4 - lq1:.4f}")
```

**Answer:** the sum collapses to $\log\big(q(x_4 \mid x_0) / q(x_1 \mid x_0)\big)$: the marginals at $t = 2$ and $t = 3$ cancel completely, and only the last and first survive. The code prints an identical sum and collapsed value.

</details>

**Problem 2.** For $T = 4$, the decomposition reads $-\mathrm{ELBO} = L_T + L_3 + L_2 + L_1 + L_0$. Which terms depend on the network parameters $\theta$, which do not, and what does that mean for training?

In [ ]:
# Your turn:

<details><summary>Show worked solution</summary>

- $L_T = D_{\mathrm{KL}}(q(x_4 \mid x_0) \,\|\, p(x_4))$: the fixed forward marginal against the fixed noise start — no $\theta$ anywhere.
- $L_3, L_2, L_1$: each is $\mathbb{E}\, D_{\mathrm{KL}}(q(x_{t-1} \mid x_t, x_0) \,\|\, p_\theta(x_{t-1} \mid x_t))$ for $t = 4, 3, 2$ — the second slot of every KL is the learned denoiser, so all three depend on $\theta$.
- $L_0 = -\mathbb{E} \log p_\theta(x_0 \mid x_1)$: contains $p_\theta$, so it depends on $\theta$.
- Adding a constant to a loss moves every value equally — it changes nothing about where the minimum sits or which way gradients point.

**Answer:** all terms except $L_T$ depend on $\theta$. $L_T$ is a schedule constant, so training minimizes $L_3 + L_2 + L_1 + L_0$ and ignores $L_T$.

</details>

**Problem 3.** The true posterior at some step is $\mathcal{N}(1.2,\ 0.09)$ and the learned denoiser predicts $\mathcal{N}(1.5,\ 0.09)$ — same variance. Compute $D_{\mathrm{KL}}$ from the first to the second, in nats.

In [ ]:
# Your turn:

<details><summary>Show worked solution</summary>

- Matched variances, so the special case applies: $D_{\mathrm{KL}} = \dfrac{(\mu_1 - \mu_2)^2}{2\sigma^2}$.
- Center gap: $1.2 - 1.5 = -0.3$; squared: $0.09$ (the sign vanishes in the square).
- Denominator: $2 \times 0.09 = 0.18$.
- Divide: $0.09 / 0.18 = 0.5$.

```python
kl_p3 = (1.2 - 1.5) ** 2 / (2 * 0.09)
check = gauss_kl(1.2, 0.09, 1.5, 0.09)

print(f"KL = {kl_p3:.4f} nats   (general formula: {check:.4f})")
```

**Answer:** $D_{\mathrm{KL}} = 0.5$ nats — a center miss of 0.3 with spread 0.09 is a substantial mismatch, several thousand times the real schedule's entire $L_T$.

</details>

**Problem 4.** For one data point, the exact log-likelihood is $\log p_\theta(x_0) = -3.2$ and the gap KL is $0.7$. What is the ELBO? A colleague's code then reports $\mathrm{ELBO} = -2.9$ for a point with $\log p_\theta(x_0) = -3.2$. What do you tell them?

In [ ]:
# Your turn:

<details><summary>Show worked solution</summary>

- The gap identity: $\log p_\theta(x_0) = \mathrm{ELBO} + \mathrm{gap}$, so $\mathrm{ELBO} = -3.2 - 0.7 = -3.9$.
- The gap is a KL, so $\mathrm{gap} \ge 0$ always, which forces $\mathrm{ELBO} \le \log p_\theta(x_0)$.
- The colleague's $-2.9$ sits **above** $-3.2$ on the number line — the reported ELBO exceeds the log-likelihood, which the math forbids.

```python
elbo_p4 = -3.2 - 0.7

print(f"ELBO = {elbo_p4:.1f} nats")
print(f"is ELBO = -2.9 possible when log p = -3.2?  {-2.9 <= -3.2}")
```

**Answer:** $\mathrm{ELBO} = -3.9$. The reported $-2.9$ is impossible — the ELBO can never exceed the log-likelihood — so the code has a bug: most likely a sign flip on the loss, or a KL computed with its arguments swapped.

</details>

**Problem 5.** Complete the comparison table between a VAE's ELBO and diffusion's ELBO, row by row: (a) what is the latent variable, (b) is the helper $q$ learned or fixed, (c) what does the generator look like, (d) when is the bound tight and how does each model get there?

In [ ]:
# Your turn:

<details><summary>Show worked solution</summary>

- (a) Latent: VAE — a single compressed code $z$, much smaller than the data; diffusion — the entire noisy path $x_{1:T}$, which is $T$ full-size frames.
- (b) Helper: VAE — a learned encoder network with its own parameters; diffusion — the fixed forward noising chain, zero parameters.
- (c) Generator: VAE — one decoder $p_\theta(x \mid z)$ that generates in a single step; diffusion — a chain of $T$ small Gaussian denoising steps sharing one network.
- (d) Tightness: the bound is tight exactly when the helper equals the model's true posterior over the latent. A VAE trains $q$ toward that posterior; diffusion cannot move $q$, so training moves $p_\theta$ until its posterior matches the fixed forward chain — Step 9 of this notebook is precisely that moment, with the gap at zero to ten decimals.

**Answer:** latent: code $z$ vs whole path $x_{1:T}$; helper: learned encoder vs fixed noising chain; generator: one-step decoder vs $T$-step Gaussian chain; tightness: both need helper = model posterior — the VAE moves the helper, diffusion moves the model.

</details>

**Problem 6.** Using the 3-state toy (keep probability 0.7, hop probability 0.15 to each other state), compute the inner KL of a middle term for $x_0 = 0$ and $x_2 = 0$, against the candidate $p_\theta(x_1 \mid x_2 = 0) = (0.6,\ 0.2,\ 0.2)$.

In [ ]:
# Your turn:

<details><summary>Show worked solution</summary>

- Forward reach: $q(x_1 \mid x_0 = 0) = (0.7,\ 0.15,\ 0.15)$ — row 0 of the kernel.
- Backward look: $q(x_2 = 0 \mid x_1) = (0.7,\ 0.15,\ 0.15)$ — column 0 of the kernel.
- Multiply elementwise: $(0.49,\ 0.0225,\ 0.0225)$; sum $= 0.535$; normalize: posterior $(0.9159,\ 0.0421,\ 0.0421)$ — the chain almost surely stayed home both steps.
- KL against $(0.6,\ 0.2,\ 0.2)$: $0.9159 \log(0.9159/0.6) + 2 \times 0.0421 \log(0.0421/0.2) = 0.3874 - 0.0656 - 0.0656 = 0.2562$ nats.

```python
numer_p6 = K[0] * K[:, 0]
post_p6 = numer_p6 / numer_p6.sum()
guess_p6 = np.array([0.6, 0.2, 0.2])
kl_p6 = np.sum(post_p6 * np.log(post_p6 / guess_p6))

print("posterior:", np.round(post_p6, 4))
print(f"KL = {kl_p6:.4f} nats")
```

**Answer:** the posterior is $(0.9159,\ 0.0421,\ 0.0421)$ and the KL is about $0.2562$ nats — the true denoiser is very confident the chain stayed at 0, so the hedging candidate pays a sizable mismatch cost.

</details>

## Wrap-up

Verified in this notebook: on a 3-state, $T = 2$ diffusion where every path is enumerable, the ELBO never exceeds the exact log-likelihood, and the gap between them equals the path-space KL to machine precision; the telescope collapses its middle marginals exactly, on every sampled path; the lesson's hand-worked posterior $(0.4516,\ 0.0968,\ 0.4516)$ and its $0.1546$-nat KL come out on the nose; the decomposition $-\mathrm{ELBO} = L_T + L_1 + L_0$ holds term by term; setting the reverse model to the true reverse conditionals closes the gap to ten decimal places, and dialing in wrongness reopens it; in the Gaussian case the per-step cost is a parabola in the guessed mean, minimized exactly at $\tilde\mu_t = 1.7066$; and on the real $T = 1000$ schedule, dropping $L_T$ costs a mere $8 \times 10^{-5}$ nats.

One target remains: teach a network to hit $\tilde\mu_t$. Part 9 pulls the loose thread — since $x_t = \sqrt{\bar\alpha_t}\, x_0 + \sqrt{1-\bar\alpha_t}\,\epsilon$, knowing the noise is the same as knowing the clean point — and the mean-matching loss collapses into the famous one-liner: **predict the noise**.